In [1]:
import os
import sys
import logging
import pandas as pd
import numpy as np


# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [2]:
# 1. Cargar y separar la data
from src.model.modelo import cargar_y_separar_datos
X_train, X_test, y_train, y_test = cargar_y_separar_datos(
    ruta_csv="../data/processed/adult_limpio.csv",
    target="income",
    test_size=0.2
)

2025-07-11 09:47:33,550 - INFO - Archivo cargado correctamente: ../data/processed/adult_limpio.csv
2025-07-11 09:47:33,551 - INFO - Dimensión total: 31572 filas, 46 columnas
2025-07-11 09:47:33,584 - INFO - Conjunto de entrenamiento: 25257 filas
2025-07-11 09:47:33,585 - INFO - Conjunto de prueba: 6315 filas


In [3]:
# 2. Balance de clases
from src.model.modelo import revisar_balance_clases, balancear_con_smote

# Revisar frecuencias de clases
revisar_balance_clases(y_train)

2025-07-11 09:47:34,009 - INFO - Balance de clases en el conjunto de entrenamiento:
2025-07-11 09:47:34,011 - INFO - 
        frecuencia  proporcion
income                        
0            19125       75.72
1             6132       24.28


In [4]:
# Realizar oversampling con SMOTE
X_train_res, y_train_res = balancear_con_smote(X_train, y_train)

2025-07-11 09:47:37,151 - INFO - Aplicado SMOTE: 38250 muestras totales después del balanceo
2025-07-11 09:47:37,154 - INFO - Distribución posterior:
income
0    19125
1    19125
Name: count, dtype: int64


In [5]:
# 3. Ajuste de hiperparámetros
from src.model.modelo import entrenar_modelos_clasificacion

# Método de búsqueda en cuadrícula
resultados = entrenar_modelos_clasificacion(
    X_train_res, y_train_res,
    metodo="grid",
    save_path="../outputs/03_modelado/grid_resumen_modelos.html"
)

2025-07-11 09:47:37,220 - INFO - Entrenando modelo: KNN (grid)
2025-07-11 09:48:06,105 - INFO - Mejor accuracy para KNN: 0.8524
2025-07-11 09:48:06,106 - INFO - Hiperparámetros óptimos: {'n_neighbors': 7, 'weights': 'distance'}
2025-07-11 09:48:06,107 - INFO - Entrenando modelo: SVM (grid)
2025-07-11 10:09:54,916 - INFO - Mejor accuracy para SVM: 0.8492
2025-07-11 10:09:55,008 - INFO - Hiperparámetros óptimos: {'C': 10, 'kernel': 'rbf'}
2025-07-11 10:09:55,011 - INFO - Entrenando modelo: LogisticRegression (grid)
2025-07-11 10:10:04,609 - INFO - Mejor accuracy para LogisticRegression: 0.8277
2025-07-11 10:10:04,611 - INFO - Hiperparámetros óptimos: {'C': 10, 'penalty': 'l2'}
2025-07-11 10:10:04,612 - INFO - Entrenando modelo: DecisionTree (grid)
2025-07-11 10:10:08,845 - INFO - Mejor accuracy para DecisionTree: 0.8392
2025-07-11 10:10:08,846 - INFO - Hiperparámetros óptimos: {'criterion': 'gini', 'max_depth': 10}
2025-07-11 10:10:08,847 - INFO - Entrenando modelo: RandomForest (grid)
2

In [6]:
# Método de búsqueda aleatoria
resultados_random = entrenar_modelos_clasificacion(
    X_train_res, y_train_res,
    metodo="random",
    save_path="../outputs/03_modelado/random_resumen_modelos.html"
)

2025-07-11 10:13:13,452 - INFO - Entrenando modelo: KNN (random)
2025-07-11 10:13:31,820 - INFO - Mejor accuracy para KNN: 0.8524
2025-07-11 10:13:31,822 - INFO - Hiperparámetros óptimos: {'weights': 'distance', 'n_neighbors': 7}
2025-07-11 10:13:31,823 - INFO - Entrenando modelo: SVM (random)
2025-07-11 10:31:19,174 - INFO - Mejor accuracy para SVM: 0.8492
2025-07-11 10:31:19,233 - INFO - Hiperparámetros óptimos: {'kernel': 'rbf', 'C': 10}
2025-07-11 10:31:19,237 - INFO - Entrenando modelo: LogisticRegression (random)
C:\Proyectos_Pycharm\Laboratorio_16_Aplicaciones_ML\.venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=5. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2025-07-11 10:31:28,951 - INFO - Mejor accuracy para LogisticRegression: 0.8277
2025-07-11 10:31:28,955 - INFO - Hiperparámetros óptimos: {'penalty': 'l2', 'C': 10}
2025-07-11 10:31:28,957 - INFO - Entrenan

CONCLUSIONES
1. El modelo con mejor desempeño fue Random Forest, alcanzando un accuracy de 86.61%, superando ligeramente a KNN (85.24%) y SVM (84.92%). Esto respalda su reputación como modelo robusto frente a distintos tipos de datos y relaciones no lineales.

2. Los cinco algoritmos probados lograron un buen desempeño, lo que indica que el preprocesamiento aplicado (transformación, estandarización, imputación, dummificación y balanceo) fue adecuado para preparar los datos.

3. Los mejores hiperparámetros fueron idénticos con ambas estrategias de búsqueda (Grid y Random), lo que sugiere que el espacio de búsqueda definido fue apropiado y acotado.

4. RandomizedSearchCV demostró ser más eficiente, encontrando los mismos resultados en menor tiempo (18 minutos frente a 23 minutos de GridSearchCV). Esto sugiere que, en futuros experimentos con espacios de búsqueda más grandes, Random Search podría ser preferido como estrategia inicial.

5. La selección de modelos con diferentes naturalezas (lineales, basados en vecinos, árboles y ensambles) fue clave para garantizar una evaluación diversa y justa del rendimiento, facilitando una mejor decisión sobre cuál usar en producción o ensamblajes posteriores.